In [15]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

def apply_sepia(image):
    sepia_matrix = np.array([
        [0.393, 0.769, 0.189],
        [0.349, 0.686, 0.168],
        [0.272, 0.534, 0.131]
    ])
    sepia_img = image.dot(sepia_matrix.T)
    return np.clip(sepia_img, 0, 1)

def generate_image_dataset(num_images=300, img_size=(32, 32, 3)):
    np.random.seed(42)
    images = np.random.rand(num_images, *img_size)
    labels = np.zeros(num_images, dtype=int) # 0 = Normal, 1 = Sepia

    for i in range(num_images // 2, num_images):
        images[i] = apply_sepia(images[i])
        labels[i] = 1

    permutation = np.random.permutation(num_images)
    return images[permutation], labels[permutation]

inputs, outputs = generate_image_dataset()
outputNames = ['Normal', 'Sepia']
print(f"Baza de date creata: {len(inputs)} imagini de dimensiune {inputs[0].shape}")

Baza de date creata: 300 imagini de dimensiune (32, 32, 3)


In [16]:
inputs_flatten = np.array([img.flatten() for img in inputs])

def splitData(inputs, outputs):
    np.random.seed(5)
    indexes = [i for i in range(len(inputs))]
    trainSample = np.random.choice(indexes, int(0.8 * len(inputs)), replace=False)
    testSample = [i for i in indexes if not i in trainSample]

    trainInputs = [inputs[i] for i in trainSample]
    trainOutputs = [outputs[i] for i in trainSample]
    testInputs = [inputs[i] for i in testSample]
    testOutputs = [outputs[i] for i in testSample]

    return trainInputs, trainOutputs, testInputs, testOutputs

trainInputs, trainOutputs, testInputs, testOutputs = splitData(inputs_flatten, outputs)

def normalisation(trainData, testData):
    scaler = StandardScaler()
    scaler.fit(trainData)
    return scaler.transform(trainData), scaler.transform(testData)

trainInputs, testInputs = normalisation(trainInputs, testInputs)
print(f"Date de antrenare: {len(trainInputs)} | Date de test: {len(testInputs)}")

Date de antrenare: 240 | Date de test: 60


In [17]:
classifier = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', max_iter=200,
                           solver='adam', random_state=1, learning_rate_init=0.001)

print("Antrenare MLPClassifier in curs...")
classifier.fit(trainInputs, trainOutputs)
predictedLabels = classifier.predict(testInputs)

print(f"Acuratete Tool: {accuracy_score(testOutputs, predictedLabels):.4f}")
print(f"Precizie Tool: {precision_score(testOutputs, predictedLabels):.4f}")
print(f"Recall Tool: {recall_score(testOutputs, predictedLabels):.4f}")

Antrenare MLPClassifier in curs...
Acuratete Tool: 1.0000
Precizie Tool: 1.0000
Recall Tool: 1.0000


In [18]:
hyperparameter_configs = [
    {'hidden_layer_sizes': (10,), 'learning_rate_init': 0.01, 'max_iter': 100},
    {'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001, 'max_iter': 200},
    {'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.001, 'max_iter': 200},
    {'hidden_layer_sizes': (50,), 'learning_rate_init': 0.1, 'max_iter': 100},
]

print(f"{'Hidden Layers':<20} | {'Learning Rate':<15} | {'Max Iter':<10} | {'Acuratete'}")
print("-" * 65)

for config in hyperparameter_configs:
    clf = MLPClassifier(
        hidden_layer_sizes=config['hidden_layer_sizes'],
        learning_rate_init=config['learning_rate_init'],
        max_iter=config['max_iter'], activation='relu', solver='adam', random_state=42
    )
    clf.fit(trainInputs, trainOutputs)
    acc = accuracy_score(testOutputs, clf.predict(testInputs))
    print(f"{str(config['hidden_layer_sizes']):<20} | {config['learning_rate_init']:<15} | {config['max_iter']:<10} | {acc:.4f}")

Hidden Layers        | Learning Rate   | Max Iter   | Acuratete
-----------------------------------------------------------------
(10,)                | 0.01            | 100        | 1.0000
(50,)                | 0.001           | 200        | 1.0000
(100, 50)            | 0.001           | 200        | 1.0000
(50,)                | 0.1             | 100        | 1.0000


In [19]:
import numpy as np

# ---------- ACTIVATIONS ----------
def relu(x):
    return np.maximum(0, x)

def derivata_relu(x):
    return (x > 0).astype(float)

# ---------- LOSS & SIGMOID ----------
class StratSigmoid:
    def forward(self, x):
        self.out = 1 / (1 + np.exp(-np.clip(x, -500, 500)))
        return self.out
    def backward(self, grad, lr):
        return grad * self.out * (1 - self.out)

def binary_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def derivata_bce(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)
    return (y_pred - y_true) / (y_pred * (1 - y_pred) * y_true.size)

# ---------- LAYERS ----------
class StratDens:
    def __init__(self, intrari, iesiri):
        self.ponderi = np.random.randn(intrari, iesiri) * np.sqrt(2. / intrari)
        self.bias = np.zeros((1, iesiri))

    def forward(self, intrare):
        self.intrare = intrare
        return np.dot(intrare, self.ponderi) + self.bias

    def backward(self, eroare_iesire, lr):
        grad_w = np.dot(self.intrare.T, eroare_iesire)
        grad_b = np.sum(eroare_iesire, axis=0, keepdims=True)
        grad_intrare = np.dot(eroare_iesire, self.ponderi.T)

        self.ponderi -= lr * grad_w
        self.bias -= lr * grad_b
        return grad_intrare

class StratActivareReLU:
    def forward(self, intrare):
        self.intrare = intrare
        return relu(intrare)
    def backward(self, grad, lr):
        return grad * derivata_relu(self.intrare)

class StratFlatten:
    def forward(self, intrare):
        self.shape = intrare.shape
        return intrare.reshape(1, -1)
    def backward(self, grad, lr):
        return grad.reshape(self.shape)

class StratMaxPool:
    def __init__(self, size=2):
        self.size = size

    def forward(self, x):
        self.x = x
        c, h, w = x.shape
        self.out = np.zeros((c, h // self.size, w // self.size))
        for i in range(0, self.out.shape[1]):
            for j in range(0, self.out.shape[2]):
                region = x[:, i*self.size:(i+1)*self.size, j*self.size:(j+1)*self.size]
                self.out[:, i, j] = np.max(region, axis=(1, 2))
        return self.out

    def backward(self, grad, lr):
        dx = np.zeros_like(self.x)
        c, h_out, w_out = grad.shape
        for i in range(h_out):
            for j in range(w_out):
                region = self.x[:, i*self.size:(i+1)*self.size, j*self.size:(j+1)*self.size]
                for ch in range(c):
                    # Gasim indexul valorii maxime in fereastra
                    idx = np.argmax(region[ch])
                    ii, jj = np.unravel_index(idx, (self.size, self.size))
                    dx[ch, i*self.size + ii, j*self.size + jj] += grad[ch, i, j]
        return dx

class StratConv2D:
    def __init__(self, in_channels, out_channels, k):
        self.k = k
        self.c_out = out_channels
        self.filtre = np.random.randn(out_channels, in_channels, k, k) * np.sqrt(2. / (in_channels * k * k))
        self.bias = np.zeros((out_channels, 1))

    def forward(self, x):
        self.x = x
        c_in, h, w = x.shape
        h_out, w_out = h - self.k + 1, w - self.k + 1
        self.out = np.zeros((self.c_out, h_out, w_out))

        for i in range(h_out):
            for j in range(w_out):
                region = x[:, i:i+self.k, j:j+self.k]
                # Operatie vectorizata pe toate filtrele simultan
                self.out[:, i, j] = np.sum(region * self.filtre, axis=(1, 2, 3)) + self.bias.flatten()
        return self.out

    def backward(self, grad, lr):
        d_filters = np.zeros_like(self.filtre)
        d_input = np.zeros_like(self.x)

        for i in range(grad.shape[1]):
            for j in range(grad.shape[2]):
                region = self.x[:, i:i+self.k, j:j+self.k]
                for f in range(self.c_out):
                    d_filters[f] += grad[f, i, j] * region
                    d_input[:, i:i+self.k, j:j+self.k] += grad[f, i, j] * self.filtre[f]

        self.filtre -= lr * d_filters
        self.bias -= lr * np.sum(grad, axis=(1, 2)).reshape(-1, 1)
        return d_input

# ---------- TRAINING LOOP ----------
def antreneaza(retea, X, Y, epoci=10, lr=0.01):
    for epoca in range(epoci):
        loss_total = 0
        correct = 0

        for x, y in zip(X, Y):
            # Forward
            out = x
            for strat in retea:
                out = strat.forward(out)

            loss_total += binary_cross_entropy(y, out)
            pred = 1 if out[0][0] > 0.5 else 0
            if pred == int(y[0]):
                correct += 1

            # Backward
            grad = derivata_bce(y, out)
            for strat in reversed(retea):
                grad = strat.backward(grad, lr)

        print(f"Epoca {epoca+1}/{epoci} | Loss: {loss_total/len(X):.4f} | Acc: {100*correct/len(X):.1f}%")

In [20]:
# Transpunem imaginile pentru formatul (C, H, W) cerut de CNN
images_cnn = inputs.transpose(0, 3, 1, 2)

# Split pe datele brute (fara flatten)
np.random.seed(42) # setam un seed pentru reproductibilitate
indices = np.random.permutation(len(images_cnn))
split_idx = int(0.8 * len(images_cnn))

train_X = images_cnn[indices[:split_idx]]
train_Y = outputs[indices[:split_idx]].reshape(-1, 1)
test_X = images_cnn[indices[split_idx:]]
test_Y = outputs[indices[split_idx:]].reshape(-1, 1)

# Definire arhitectura
model = [
    StratConv2D(in_channels=3, out_channels=4, k=3), # Output: 4 x 30 x 30
    StratActivareReLU(),
    StratMaxPool(size=2),                            # Output: 4 x 15 x 15
    StratFlatten(),                                  # Output: 1 x (4*15*15) = 1 x 900
    StratDens(900, 20),
    StratActivareReLU(),
    StratDens(20, 1),
    StratSigmoid()
]

# Antrenare
print("Începe antrenarea CNN-ului propriu...")
antreneaza(model, train_X, train_Y)

input_size_ann = 32 * 32 * 3

# Definire arhitectura
model_ann_propriu = [
    StratFlatten(),                      # 1. Aplatizăm direct imaginea (3072 de valori)
    StratDens(input_size_ann, 100),      # 2. Strat ascuns complet conectat cu 100 neuroni
    StratActivareReLU(),                 # 3. Activare non-liniară
    StratDens(100, 1),                   # 4. Strat de ieșire (1 neuron pentru decizie 0 sau 1)
    StratSigmoid()                       # 5. Forțăm ieșirea între 0 și 1 (Probabilitate)
]

# Antrenăm ANN-ul
print("Începe antrenarea ANN-ului propriu...")
antreneaza(model_ann_propriu, train_X, train_Y, epoci=10, lr=0.01)

Începe antrenarea CNN-ului propriu...
Epoca 1/10 | Loss: 0.2780 | Acc: 92.1%
Epoca 2/10 | Loss: 0.0089 | Acc: 100.0%
Epoca 3/10 | Loss: 0.0043 | Acc: 100.0%
Epoca 4/10 | Loss: 0.0027 | Acc: 100.0%
Epoca 5/10 | Loss: 0.0020 | Acc: 100.0%
Epoca 6/10 | Loss: 0.0016 | Acc: 100.0%
Epoca 7/10 | Loss: 0.0013 | Acc: 100.0%
Epoca 8/10 | Loss: 0.0011 | Acc: 100.0%
Epoca 9/10 | Loss: 0.0009 | Acc: 100.0%
Epoca 10/10 | Loss: 0.0008 | Acc: 100.0%
Începe antrenarea ANN-ului propriu...
Epoca 1/10 | Loss: 0.6635 | Acc: 62.1%
Epoca 2/10 | Loss: 0.1820 | Acc: 96.7%
Epoca 3/10 | Loss: 0.0232 | Acc: 100.0%
Epoca 4/10 | Loss: 0.0110 | Acc: 100.0%
Epoca 5/10 | Loss: 0.0070 | Acc: 100.0%
Epoca 6/10 | Loss: 0.0050 | Acc: 100.0%
Epoca 7/10 | Loss: 0.0039 | Acc: 100.0%
Epoca 8/10 | Loss: 0.0031 | Acc: 100.0%
Epoca 9/10 | Loss: 0.0026 | Acc: 100.0%
Epoca 10/10 | Loss: 0.0022 | Acc: 100.0%


In [21]:
def testeaza(retea, X_test, Y_test):
    correct = 0
    predicții = []

    for x, y in zip(X_test, Y_test):
        out = x
        for strat in retea:
            out = strat.forward(out)

        pred = 1 if out[0][0] > 0.5 else 0
        predicții.append(pred)

        if pred == int(y[0]):
            correct += 1

    acuratete = 100 * correct / len(X_test)
    print(f"Acuratete CNN propriu: {acuratete:.2f}%")
    return predicții

# Rulare test
print("Se testează modelul CNN propriu...")
preds = testeaza(model, test_X, test_Y)

print("\nSe testează modelul ANN propriu...")
preds_ann = testeaza(model_ann_propriu, test_X, test_Y)

Se testează modelul CNN propriu...
Acuratete CNN propriu: 100.00%

Se testează modelul ANN propriu...
Acuratete CNN propriu: 100.00%
